In [15]:
!pip install sqlalchemy 
!pip install cryptography
!pip install pymysql

In [16]:
import pandas as pd
from sqlalchemy import create_engine
import getpass # Para pedir la contraseña de forma segura

In [17]:
# --- Variables de Conexión (Asegúrate de cambiar estos valores por los tuyos) ---
# Si tu base de datos no es MySQL, el "mysql+..." y el puerto (3306) deben cambiar.
DB_TIPO = 'mysql'
DB_DRIVER = 'pymysql' # O 'mysqlclient' si lo tienes instalado
DB_USER = 'root'      # Tu nombre de usuario
DB_PASS = getpass.getpass("Ingresa tu contraseña de MySQL: ") # Te pedirá la contraseña
DB_HOST = '127.0.0.1' # O la IP donde esté tu DB
DB_PORT = '3306'      # Puerto por defecto de MySQL
DB_NAME = 'sakila'    # El nombre de la base de datos

# --- Creación del Motor de Conexión (Engine) ---
# Esta línea construye la URL de conexión que usa SQLAlchemy
mysql_url = f"{DB_TIPO}+{DB_DRIVER}://{DB_USER}:{DB_PASS}@{DB_HOST}:{DB_PORT}/{DB_NAME}"

# Creamos el motor. Este objeto 'engine' es el que usaremos para todas las consultas.
try:
    engine = create_engine(mysql_url)
    print("✅ Conexión establecida con éxito.")
except Exception as e:
    print(f"❌ Error al conectar a la base de datos: {e}")
    # Es crucial que la conexión funcione antes de seguir.

✅ Conexión establecida con éxito.


In [18]:
# PASO 2
def rentals_month(engine, month: int, year: int) -> pd.DataFrame:
    """
    Recupera todos los registros de alquiler para un mes y año específicos 
    de la tabla 'rental' de la base de datos Sakila.

    Parámetros:
    - engine: Objeto de conexión de SQLAlchemy.
    - month (int): El número del mes (ej: 5 para Mayo).
    - year (int): El número del año (ej: 2005).

    Retorna:
    - pd.DataFrame: Un DataFrame de Pandas con los resultados.
    """
    
    # 1. Creamos la consulta SQL
    # La consulta filtra por el mes y el año de la columna 'rental_date'.
    query = f"""
    SELECT 
        rental_id, 
        customer_id, 
        rental_date
    FROM 
        rental
    WHERE 
        MONTH(rental_date) = {month} AND YEAR(rental_date) = {year};
    """
    
    print(f"Buscando alquileres para el mes {month}/{year}...")
    
    # 2. Ejecutamos la consulta usando el engine y pandas.read_sql
    # Esto es la magia: pandas se conecta, ejecuta la query y trae el resultado 
    # directamente como un DataFrame.
    df = pd.read_sql(query, engine)
    
    print(f"Se encontraron {len(df)} alquileres.")
    
    return df


In [19]:
def rental_count_month(df: pd.DataFrame, month: int, year: int) -> pd.DataFrame:
    column_name = f"rentals_{month:02d}_{year}" 
    rental_counts = df.groupby('customer_id').size().reset_index(name=column_name)
    return rental_counts


In [20]:
## PASO 4: Función para comparar la actividad (merge)

def compare_rentals(df1: pd.DataFrame, df2: pd.DataFrame) -> pd.DataFrame:
    # Obtener nombres de columnas dinámicamente
    col1 = [col for col in df1.columns if col != 'customer_id'][0] 
    col2 = [col for col in df2.columns if col != 'customer_id'][0] 
    
    # Inner merge
    combined_df = pd.merge(df1, df2, on='customer_id', how='inner')
    
    # Calcular diferencia: Segundo mes - Primer mes
    combined_df['difference'] = combined_df[col2] - combined_df[col1]

    # Reordenar
    combined_df = combined_df[['customer_id', col1, col2, 'difference']]
    
    return combined_df

In [21]:
# EJECUCIÓN 

# 1. Recuperar datos
df_may = rentals_month(engine, month=5, year=2005)
df_jun = rentals_month(engine, month=6, year=2005)

# 2. Contar alquileres por cliente
df_may_counts = rental_count_month(df_may, month=5, year=2005)
df_jun_counts = rental_count_month(df_jun, month=6, year=2005)

# 3. Comparar
final_comparison = compare_rentals(df_may_counts, df_jun_counts)

print("\nResultados de la comparación (Mayo vs. Junio):")
print(final_comparison.sort_values(by='difference', ascending=False).head(10))

Buscando alquileres para el mes 5/2005...
Se encontraron 1156 alquileres.
Buscando alquileres para el mes 6/2005...
Se encontraron 2311 alquileres.

Resultados de la comparación (Mayo vs. Junio):
     customer_id  rentals_05_2005  rentals_06_2005  difference
386          454                1               10           9
178          213                1                9           8
248          295                1                9           8
389          457                1                9           8
322          380                1                8           7
194          234                1                8           7
218          260                1                8           7
481          561                2                9           7
23            27                1                8           7
224          267                3                9           6
